In [1]:
from typing import Any, Dict, Tuple

import numpy as np
import torch
from torch import nn
from lightning import LightningModule
from torchmetrics import MaxMetric, MeanMetric
from torchmetrics.classification.accuracy import Accuracy
from torch.cuda.amp import GradScaler, autocast

from monai.inferers import sliding_window_inference
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.networks.nets import SwinUNETR 
from monai.transforms import Activations, AsDiscrete, Compose
from monai.utils.enums import MetricReduction
from monai.data import decollate_batch

import numpy as np
import time
import os
import shutil

from functools import partial
import wandb

    


/work/hpc/miniconda3/envs/mri/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [ ]:
# Manual dice score
def dice(x, y):
    intersect = np.sum(np.sum(np.sum(x * y)))
    y_sum = np.sum(np.sum(np.sum(y)))
    if y_sum == 0:
        return 0.0
    x_sum = np.sum(np.sum(np.sum(x)))
    return 2 * intersect / (x_sum + y_sum)

# Spherical Laplacian Gradient 3D Window
def construct_window(radius, mode='cyclic', weight= 1.):
    """ Constructing window for non-equal supression
    """
    assert radius % 2 != 0
    r = radius // 2

    # add noise to prevent neighbor exclusion
    window = np.ones([radius, radius, radius]) 
    x = np.linspace(-r, r, radius)
    y, z = x.copy(), x.copy()
    xv, yv, zv = np.meshgrid(x, y, z)

    if mode == 'cyclic':
        window[xv**2 + yv**2 + zv**2 >= (r + 0.25) ** 2] = 0
    elif mode == 'cross':
        window *= (xv == 0) | (yv == 0) | (zv == 0)
    elif mode == 'uni-cross':
        x = xv == 0
        y = yv == 0
        z = zv == 0
        window *= (x & y) | (y & z) | (z & x)
    window[r][r][r] = 0
    window /= np.sum(window) 
    window *= weight
    window[r][r][r] = -np.sum(window)

    return window

def laplacian(channel, radius=3, mode='cyclic'):
    conv = 0
    weight = construct_window(radius, mode=mode)
    conv = nn.Conv3d(   in_channel=channel, 
                        out_channel=channel, 
                        padding=channel // 2, 
                        groups=1)
    

In [ ]:
conv = nn.Conv3d(   in_channels=3, 
                    out_channels=3, 
                    kernel_size=3,
                    stride=1,
                    dilation=1,
                    padding=1, 
                    groups=1)

In [ ]:
conv.weight

In [ ]:
conv_weight = construct_window(3, mode='cyclic', weight= 2.)

In [ ]:
conv_weight

In [ ]:
conv.weight = torch.nn.Parameter(torch.from_numpy(conv_weight))

In [ ]:
conv.weight.requires_grad = False

In [ ]:
conv.weight

In [ ]:
class Lambda(nn.Module):
    def __init__(self, lambd):
        super().__init__()
        import types
        assert type(lambd) is types.LambdaType
        self.lambd = lambd

    def forward(self, x):
        return self.lambd(x)

In [ ]:
normal = lambda matrix: torch.linalg.norm(matrix, ord= None, dim=1, keepdim=True)

In [ ]:
norm = Lambda(normal)

In [ ]:
border_extractor = nn.Sequential(conv, norm)

In [ ]:
cd ..

In [2]:
from omegaconf import DictConfig
import hydra
import rootutils
rootutils.setup_root(search_from=".", indicator='setup.py', pythonpath=True)

PosixPath('/work/hpc/spine-segmentation')

In [3]:

with hydra.initialize(version_base="1.3", config_path="../configs", ):
    cfg = hydra.compose(config_name='train.yaml')
    print(cfg)

{'task_name': 'train', 'tags': ['dev'], 'train': True, 'test': True, 'ckpt_path': None, 'seed': None, 'data': {'transform_train': {'_target_': 'monai.transforms.Compose', 'transforms': [{'_target_': 'monai.transforms.LoadImaged', 'keys': ['image', 'label']}, {'_target_': 'src.data.transforms.array.ConvertToMultiChannelBasedOnSpiderClassesdSemantic', 'keys': 'label'}, {'_target_': 'monai.transforms.EnsureChannelFirstd', 'keys': 'image'}, {'_target_': 'monai.transforms.Spacingd', 'keys': ['image', 'label'], 'pixdim': [1.7, 0.625, 0.58742571], 'mode': 3}, {'_target_': 'monai.transforms.CropForegroundd', 'keys': ['image', 'label'], 'source_key': 'image', 'k_divisible': 32}, {'_target_': 'monai.transforms.RandSpatialCropd', 'keys': ['image', 'label'], 'roi_size': [32, 288, 288], 'random_size': False}, {'_target_': 'monai.transforms.RandFlipd', 'keys': ['image', 'label'], 'prob': 0.5, 'spatial_axis': 0}, {'_target_': 'monai.transforms.RandFlipd', 'keys': ['image', 'label'], 'prob': 0.5, 'spa

In [4]:
cfg.data.batch_size = 2

In [5]:
datamodule = hydra.utils.instantiate(cfg.data)

monai.transforms.croppad.dictionary CropForegroundd.__init__:allow_smaller: Current default value of argument `allow_smaller=True` has been deprecated since version 1.2. It will be changed to `allow_smaller=False` in version 1.5.


In [ ]:
datamodule.setup()

In [ ]:
dataloader = datamodule.train_dataloader()

In [ ]:
from src.models.spider_semantic_module import  SpiderLitModule

In [ ]:
model = SpiderLitModule.load_from_checkpoint("/work/hpc/spine-segmentation/logs/train/runs/attention-unet-v2/checkpoints/epoch_271_v2.ckpt")

In [ ]:
iterator = iter(dataloader)

In [ ]:
batch = next(iterator)

In [ ]:
!export CUDA_VISIBLE_DEVICES=0

In [ ]:
net = model.net

In [ ]:
criterion = model.criterion 

In [ ]:
net.to("cuda:0")

In [ ]:
x = batch['image']
x.size()

In [ ]:
x.to("cuda:0")

In [ ]:
type(x)

In [ ]:
model.to("cuda:0")

In [ ]:
x = x.cuda()

In [ ]:
logits = net(x)

In [ ]:
logits.size()

In [ ]:
y = batch['label'].cuda().to("cuda:0")
criterion.to("cuda:0")

In [ ]:
loss = criterion(logits, y)

In [ ]:
loss

In [ ]:
y.size()

In [ ]:
border_extractor.to("cuda:0")

In [ ]:
border = border_extractor(y)

In [ ]:
border.size()

In [ ]:
visual = border.detach().cpu().numpy()

In [ ]:
visual.shape

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
sample = np.sum(visual[0], axis=1)[0]
sample.shape

In [ ]:
visual[visual < 0.1] = 0

In [ ]:
values = np.unique(visual)

In [ ]:
values 

In [ ]:
plt.legend()
plt.imshow(visual[0, 0, :, 201])

In [ ]:
print(visual[0,0,15,201,70])

In [ ]:
import SimpleITK as sitk 

In [ ]:
volume = sitk.GetImageFromArray(visual[0, 0])

In [ ]:
writer = sitk.ImageFileWriter()
writer.SetFileName("/work/hpc/spine-segmentation/outputs/dummy/border.nii.gz")
writer.Execute(volume)

In [ ]:
model.cpu()

In [ ]:
x.detach().cpu()

In [ ]:
y.detach().cpu()

In [ ]:
logits.detach().cpu()

In [ ]:
del x, y, logits, model, border_extractor